<h3> Crawling the headlines and contents from Investing.com "Commodities" section

In [ ]:
import logging
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import os

logging.basicConfig(
    format='%(asctime)s %(levelname)s:%(message)s',
    level=logging.INFO)

class Crawler:
    def __init__(self, urls=[]):
        self.visited_urls = []
        self.urls_to_visit = urls

    def download_url(self, url):
        return requests.get(url).text

    def get_linked_urls(self, url, html):
        soup = BeautifulSoup(html, 'html.parser')
        container = soup.find('div', class_="news-analysis-v2_articles-container__3fFL8 mdMax:px-3 mb-12")  
        if container:
            for link in container.find_all('a'):  
                path = link.get('href')
                if path and path.startswith('/'):
                    path = urljoin(url, path)
                    yield path
        else:
            logging.info(f'No container with class found on {url}')

    def add_url_to_visit(self, url):
        if url not in self.visited_urls and url not in self.urls_to_visit:
            self.urls_to_visit.append(url)

    def crawl(self, url):
        html = self.download_url(url)
        for url in self.get_linked_urls(url, html):
            self.add_url_to_visit(url)

    def run(self):
        while self.urls_to_visit:
            url = self.urls_to_visit.pop(0)
            logging.info(f'Crawling: {url}')
            try:
                self.crawl(url)
            except Exception:
                logging.exception(f'Failed to crawl: {url}')
            finally:
                self.visited_urls.append(url)

if __name__ == '__main__':
    base_url = 'https://www.investing.com/news/commodities-news/'
    urls = [f"{base_url}{page}" for page in range(450, 565)] # How many pages would you like to crawl
    # Edit this row before running the program
    C = Crawler(urls)
    C.run()

    linkagelst = []
    for url in C.visited_urls:
        linkagelst.append(url)
    linkagelst = [item for item in linkagelst if (item.startswith("https://www.investing.com/news/commodities-news/"))]
    linkagelst = [item for item in linkagelst if not (item.endswith("#comments"))]
    numberofWebpages = 115 # Edit this row before running the program
    for i in range (numberofWebpages): # Remove the first n linkages according to the number of webpages you crawl
        linkagelst.pop(0)
    print(len(linkagelst))
    print(linkagelst)

    def getContent(url):
        response = requests.get(url)
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'html.parser')
            titleElement = soup.find(id="articleTitle")
            dateElement = soup.find('div', class_="flex flex-col gap-2 text-warren-gray-700 md:flex-row md:items-center md:gap-0")
            if titleElement and dateElement:
                title = titleElement.text.strip()
                date = soup.find_all('div', {'class': 'flex flex-row items-center'})[1].span.text
                container = soup.find('div', class_='article_container')
                descriptionElements = container.find_all('p')
                description = ' '.join(p.text.strip() for p in descriptionElements)
                return [title, date, description]
        else:
            print(f"Failed to retrieve content from {url}. Status code: {response.status_code}")
            return None
        
    path = "decoy"

    for linkage in linkagelst:
        url = linkage
        content = getContent(url)

        if content:
            title, date, content = content
            sanitizedTitle = title.replace('/', '-').replace('\\', '-').replace(':', '-').replace('*', '-').replace('?', '-').replace('"', '-').replace('<', '-').replace('>', '-').replace('|', '-')
            date = date.replace(',','')
            firstSplit = date.split(' ')
            secondSplit = firstSplit[1].split("/")
            formatedDate = secondSplit[2] + secondSplit[0] + secondSplit[1]

            filename = os.path.join(path, f"{formatedDate}_{sanitizedTitle}.txt")
            with open(filename, 'w', encoding='utf-8') as f:
                f.write(f"Title: {title}\nDate: {formatedDate}\nContent:\n{content}\n")
            logging.info(f"Saved article to {filename}")

<h4> Checking whether the crawled file is empty or not

In [ ]:
import os

def listEmptyFiles(directory):
    emptyFiles = []
    for filename in os.listdir(directory):
        sourcePath = os.path.join(directory, filename)

        if os.path.isfile(sourcePath):
            with open(sourcePath, 'r', encoding='utf-8') as file:
                content = file.read().strip()
                if not content or (len(content.splitlines()) == 4 and content.splitlines()[3].strip().lower() == 'none'):
                    emptyFiles.append(filename)
    return emptyFiles

path = "decoy"
emptyFiles = listEmptyFiles(path)

for emptyFile in emptyFiles:
    print(emptyFile)